# Sri Lanka District Poverty Risk Model
### Rule-Based Risk Indexing + NLP Query Engine (Sentence Transformers)
---
**Pipeline:**
1. Load & preprocess district poverty/expenditure data
2. Engineer risk features & compute a composite Risk Index (rule-based)
3. Define allocation rules per risk tier
4. Encode semantic summaries with Sentence Transformers
5. Pack everything into one `PovertyRiskModel` and pickle it
6. Query engine — ask natural-language questions and get district-level answers

## Step 0 — Install Dependencies

In [ ]:
!pip install -q sentence-transformers==5.2.3 scikit-learn openpyxl pandas numpy
import sys

In [ ]:
!python --version

Python 3.12.12


In [ ]:
!pip show sentence-transformers
!pip show transformers
!pip show torch

Name: sentence-transformers
Version: 5.2.3
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 
Name: transformers
Version: 5.0.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Step 1 — Load & Upload Data

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np


filename = "/content/drive/MyDrive/DSGP/poverty/Povertylines.xlsx"
df_raw = pd.read_excel(filename)
df_raw.head(3)

,District,2024 jan,2024 feb,2024 mar,2024 apr,2024 may,2024 june,2024 july,2024 aug,2024 sep,...,2025 sep,mean_household_income_per_month,median_household_income_per_month_rs,average_household_size,gini_coefficient_income,mean_per_capita_income_per_month_rs,mean_household_expenditure_per_month_rs,median_household_expenditure_per_month_rs,gini_coefficient_expenditure,mean_household_per_capita_expenditure_per_month
0,Ampara,17150,17110,16752,16607,16456,16599,16504,16281,16201,...,16544,60474,42236,4.0,0.45,15856,52924,42587,0.36,13876
1,Anuradhapura,16604,16566,16219,16079,15933,16071,15979,15763,15686,...,16018,64409,46379,3.5,0.44,18473,52796,43362,0.35,15143
2,Badulla,17140,17101,16742,16598,16447,16590,16495,16272,16192,...,16535,66413,40063,3.6,0.53,18249,49971,34471,0.41,12907


## Step 2 — Feature Engineering & Risk Index

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Identify poverty-line columns (monthly snapshots) ──────────────────────
poverty_cols = [c for c in df_raw.columns if any(y in str(c) for y in ['2024', '2025'])]

df = df_raw.copy()
df.columns = df.columns.str.strip()

# ── Core socioeconomic features ────────────────────────────────────────────
feature_cols = [
    'mean_household_income_per_month',
    'median_household_income_per_month_rs',
    'average_household_size',
    'gini_coefficient_income',
    'mean_per_capita_income_per_month_rs',
    'mean_household_expenditure_per_month_rs',
    'median_household_expenditure_per_month_rs',
    'gini_coefficient_expenditure',
    'mean_household_per_capita_expenditure_per_month'
]

# ── Derived Features ───────────────────────────────────────────────────────
df['poverty_line_latest']    = df[poverty_cols].iloc[:, -1]   # most recent month
df['poverty_line_mean']      = df[poverty_cols].mean(axis=1)
df['poverty_line_trend']     = df[poverty_cols].iloc[:, -1] - df[poverty_cols].iloc[:, 0]  # price drift
df['income_expenditure_gap'] = df['mean_household_income_per_month'] - df['mean_household_expenditure_per_month_rs']
df['affordability_ratio']    = df['mean_household_per_capita_expenditure_per_month'] / df['poverty_line_latest']
df['income_inequality']      = df['mean_household_income_per_month'] - df['median_household_income_per_month_rs']

extended_features = feature_cols + [
    'poverty_line_latest',
    'poverty_line_mean',
    'poverty_line_trend',
    'income_expenditure_gap',
    'affordability_ratio',
    'income_inequality'
]

print('Features used for risk scoring:')
for f in extended_features:
    print(f'  • {f}')

Features used for risk scoring:
  • mean_household_income_per_month
  • median_household_income_per_month_rs
  • average_household_size
  • gini_coefficient_income
  • mean_per_capita_income_per_month_rs
  • mean_household_expenditure_per_month_rs
  • median_household_expenditure_per_month_rs
  • gini_coefficient_expenditure
  • mean_household_per_capita_expenditure_per_month
  • poverty_line_latest
  • poverty_line_mean
  • poverty_line_trend
  • income_expenditure_gap
  • affordability_ratio
  • income_inequality


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# ── Scale all features to [0,1] ────────────────────────────────────────────
X = df[extended_features].copy()
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=extended_features, index=df.index)

# ── Rule-Based Risk Index Weights ─────────────────────────────────────────
# Higher weight = more critical poverty indicator
# Negative weight = higher value → LOWER risk
RISK_WEIGHTS = {
    'mean_household_income_per_month':                  -0.15,  # ↑ income → ↓ risk
    'median_household_income_per_month_rs':             -0.10,  # ↑ median → ↓ risk
    'average_household_size':                            0.05,  # ↑ size → ↑ burden
    'gini_coefficient_income':                           0.15,  # ↑ inequality → ↑ risk
    'mean_per_capita_income_per_month_rs':              -0.10,  # ↑ per-capita income → ↓ risk
    'mean_household_expenditure_per_month_rs':          -0.05,  # ↑ expenditure capacity → ↓ risk
    'median_household_expenditure_per_month_rs':        -0.05,  # ↑ median expenditure → ↓ risk
    'gini_coefficient_expenditure':                      0.10,  # ↑ expenditure inequality → ↑ risk
    'mean_household_per_capita_expenditure_per_month':  -0.05,  # ↑ per-capita exp → ↓ risk
    'poverty_line_latest':                               0.12,  # ↑ poverty threshold → ↑ vulnerability
    'poverty_line_mean':                                 0.05,  # sustained high threshold
    'poverty_line_trend':                                0.08,  # rising prices → ↑ risk
    'income_expenditure_gap':                           -0.08,  # ↑ gap (surplus) → ↓ risk
    'affordability_ratio':                              -0.07,  # ↑ affordability → ↓ risk
    'income_inequality':                                 0.10,  # ↑ mean-median gap → ↑ risk
}

# ✅ Verify: positives = 0.65, negatives = -0.65, total = 0.00
assert abs(sum(RISK_WEIGHTS.values())) < 0.01, 'Weights must sum to ~0 (balanced scale)'
# ── Compute weighted risk score ────────────────────────────────────────────
risk_score = pd.Series(0.0, index=df.index)
for feat, weight in RISK_WEIGHTS.items():
    if weight > 0:
        risk_score += weight * X_scaled[feat]           # positive → adds risk
    else:
        risk_score += abs(weight) * (1 - X_scaled[feat])  # negative → inverted

# Normalise final risk index to 0-100
risk_score_norm = (risk_score - risk_score.min()) / (risk_score.max() - risk_score.min()) * 100
df['risk_index'] = risk_score_norm.round(2)

# ── Risk Tier Assignment ───────────────────────────────────────────────────
def assign_tier(score):
    if score >= 70:   return 'CRITICAL'
    elif score >= 50: return 'HIGH'
    elif score >= 30: return 'MODERATE'
    else:             return 'LOW'

df['risk_tier'] = df['risk_index'].apply(assign_tier)

print(df[['District', 'risk_index', 'risk_tier']].sort_values('risk_index', ascending=False).to_string(index=False))

    District  risk_index risk_tier
  Mullaitivu      100.00  CRITICAL
     Badulla       89.07  CRITICAL
      Ampara       87.50  CRITICAL
      Jaffna       84.62  CRITICAL
      Mannar       84.58  CRITICAL
  Batticaloa       83.76  CRITICAL
   Ratnapura       82.19  CRITICAL
 Trincomalee       75.81  CRITICAL
     Kegalle       75.24  CRITICAL
Nuwara Eliya       74.94  CRITICAL
  Monaragala       65.51      HIGH
      Matale       60.91      HIGH
       Kandy       60.11      HIGH
  Hambantota       60.00      HIGH
 Polonnaruwa       57.58      HIGH
    Kalutara       55.72      HIGH
       Galle       55.52      HIGH
Anuradhapura       52.51      HIGH
  Kurunegala       50.97      HIGH
     Gampaha       45.94  MODERATE
 Kilinochchi       44.78  MODERATE
    Puttalam       41.81  MODERATE
      Matara       39.21  MODERATE
    Vavuniya       34.85  MODERATE
     Colombo        0.00       LOW


## Step 3 — Allocation Rules (Rule-Based Strategy)

In [ ]:
# ── Allocation Rules per Risk Tier ─────────────────────────────────────────
ALLOCATION_RULES = {
    'CRITICAL': {
        'description': 'Immediate intervention required. District is severely below poverty threshold with high inequality.',
        'subsidy_priority': 'HIGH',
        'cash_transfer_eligible': True,
        'food_assistance': True,
        'livelihood_program': True,
        'monitoring_frequency': 'Monthly',
        'budget_allocation_pct': 0.35,
        'recommended_actions': [
            'Emergency food & nutrition support',
            'Direct cash transfers (Aswesuma/Samurdhi)',
            'Livelihood skill training programs',
            'Healthcare subsidies',
            'Monthly household monitoring'
        ]
    },
    'HIGH': {
        'description': 'High vulnerability. Expenditure close to poverty line with significant inequality risk.',
        'subsidy_priority': 'MEDIUM-HIGH',
        'cash_transfer_eligible': True,
        'food_assistance': True,
        'livelihood_program': True,
        'monitoring_frequency': 'Quarterly',
        'budget_allocation_pct': 0.30,
        'recommended_actions': [
            'Conditional cash transfer programs',
            'Subsidised essential goods distribution',
            'SME micro-finance access',
            'Education support for children',
            'Quarterly household surveys'
        ]
    },
    'MODERATE': {
        'description': 'Moderate risk. Some households above poverty line but vulnerable to economic shocks.',
        'subsidy_priority': 'MEDIUM',
        'cash_transfer_eligible': False,
        'food_assistance': False,
        'livelihood_program': True,
        'monitoring_frequency': 'Bi-Annual',
        'budget_allocation_pct': 0.20,
        'recommended_actions': [
            'Vocational training programs',
            'Agricultural / fishery development',
            'Insurance scheme enrollment',
            'Financial literacy programs',
            'Bi-annual progress reviews'
        ]
    },
    'LOW': {
        'description': 'Low risk. District shows above-average income and expenditure relative to poverty line.',
        'subsidy_priority': 'LOW',
        'cash_transfer_eligible': False,
        'food_assistance': False,
        'livelihood_program': False,
        'monitoring_frequency': 'Annual',
        'budget_allocation_pct': 0.15,
        'recommended_actions': [
            'Preventive social safety net maintenance',
            'Economic growth incentive programs',
            'Annual performance evaluation',
            'Export & entrepreneurship support'
        ]
    }
}

print('Allocation Rules defined for tiers:', list(ALLOCATION_RULES.keys()))

Allocation Rules defined for tiers: ['CRITICAL', 'HIGH', 'MODERATE', 'LOW']


## Step 4 — Sentence Transformer Encoding (NLP Layer)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load lightweight multilingual model
encoder = SentenceTransformer('all-MiniLM-L6-v2')

def build_district_summary(row):
    """Create rich natural-language summary for each district."""
    tier  = row['risk_tier']
    rules = ALLOCATION_RULES[tier]
    actions = ', '.join(rules['recommended_actions'])
    return (
        f"{row['District']} district has a poverty risk index of {row['risk_index']:.1f} out of 100, "
        f"classified as {tier} risk. "
        f"Mean household income is Rs {row['mean_household_income_per_month']:,.0f} per month. "
        f"Mean household expenditure is Rs {row['mean_household_expenditure_per_month_rs']:,.0f} per month. "
        f"Per-capita expenditure is Rs {row['mean_household_per_capita_expenditure_per_month']:,.0f}. "
        f"Average household size is {row['average_household_size']:.1f} persons. "
        f"Gini coefficient (income) is {row['gini_coefficient_income']:.2f}. "
        f"Latest poverty line threshold is Rs {row['poverty_line_latest']:,.0f}. "
        f"Poverty line trend (price drift) is Rs {row['poverty_line_trend']:,.0f}. "
        f"Affordability ratio (per-capita exp / poverty line) is {row['affordability_ratio']:.2f}. "
        f"Policy: {rules['description']} "
        f"Monitoring: {rules['monitoring_frequency']}. "
        f"Recommended interventions: {actions}."
    )

df['nlp_summary'] = df.apply(build_district_summary, axis=1)

print('Encoding district summaries with Sentence Transformer...')
corpus_embeddings = encoder.encode(df['nlp_summary'].tolist(), show_progress_bar=True, convert_to_numpy=True)
print(f'Embeddings shape: {corpus_embeddings.shape}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding district summaries with Sentence Transformer...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (25, 384)


##  Step 5 — Build & Pack PovertyRiskModel

In [ ]:
import pickle
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class PovertyRiskModel:
    """
    Unified Poverty Risk Model.
    Encapsulates:
      • scaler         — MinMaxScaler fitted on feature matrix
      • feature_cols   — ordered list of feature column names
      • rules          — allocation rules dict keyed by risk tier
      • encoder        — SentenceTransformer for NLP query matching
      • district_data  — full enriched DataFrame (risk index, tier, summaries)
      • embeddings     — pre-computed corpus embeddings
    """

    def __init__(self, scaler, feature_cols, rules, encoder, district_data, embeddings):
        self.scaler        = scaler
        self.feature_cols  = feature_cols
        self.rules         = rules
        self.encoder       = encoder
        self.district_data = district_data
        self.embeddings    = embeddings

    # ── Risk Scoring ──────────────────────────────────────────────────────
    def predict_risk(self, district_name):
        """Return risk index and tier for a named district."""
        row = self.district_data[self.district_data['District'].str.lower() == district_name.lower()]
        if row.empty:
            return None
        r = row.iloc[0]
        return {
            'district':    r['District'],
            'risk_index':  r['risk_index'],
            'risk_tier':   r['risk_tier'],
            'allocation':  self.rules[r['risk_tier']]
        }

    # ── NLP Query Engine ──────────────────────────────────────────────────
    def query(self, user_input: str, top_k: int = 3, verbose: bool = True) -> list:
        """
        Accepts a natural-language query and returns the top-k
        most semantically relevant district poverty profiles.

        Parameters
        ----------
        user_input : str   — free-form question or keyword phrase
        top_k      : int   — number of districts to return (default 3)
        verbose    : bool  — print formatted output (default True)

        Returns
        -------
        list of dicts with district info and similarity score
        """
        q_emb   = self.encoder.encode([user_input], convert_to_numpy=True)
        scores  = cosine_similarity(q_emb, self.embeddings)[0]
        top_idx = np.argsort(scores)[::-1][:top_k]

        results = []
        for rank, idx in enumerate(top_idx, 1):
            row   = self.district_data.iloc[idx]
            tier  = row['risk_tier']
            rule  = self.rules[tier]
            entry = {
                'rank':         rank,
                'district':     row['District'],
                'risk_index':   row['risk_index'],
                'risk_tier':    tier,
                'similarity':   round(float(scores[idx]), 4),
                'income':       row['mean_household_income_per_month'],
                'expenditure':  row['mean_household_expenditure_per_month_rs'],
                'per_capita_exp': row['mean_household_per_capita_expenditure_per_month'],
                'gini_income':  row['gini_coefficient_income'],
                'poverty_line': row['poverty_line_latest'],
                'affordability_ratio': round(row['affordability_ratio'], 3),
                'actions':      rule['recommended_actions'],
                'monitoring':   rule['monitoring_frequency'],
                'cash_eligible': rule['cash_transfer_eligible'],
                'summary':      row['nlp_summary']
            }
            results.append(entry)

            if verbose:
                self._print_result(entry)

        return results

    # ── Ranked Risk Table ─────────────────────────────────────────────────
    def risk_table(self):
        """Return DataFrame of all districts ranked by risk index."""
        cols = ['District', 'risk_index', 'risk_tier',
                'mean_household_income_per_month',
                'mean_household_per_capita_expenditure_per_month',
                'gini_coefficient_income', 'affordability_ratio']
        return (self.district_data[cols]
                    .sort_values('risk_index', ascending=False)
                    .reset_index(drop=True))

    # ── Tier Filter ───────────────────────────────────────────────────────
    def filter_by_tier(self, tier: str):
        """Return all districts in a given risk tier."""
        tier = tier.upper()
        sub  = self.district_data[self.district_data['risk_tier'] == tier]
        return sub[['District', 'risk_index', 'mean_household_income_per_month',
                    'gini_coefficient_income']].reset_index(drop=True)

    # ── Pretty Printer ────────────────────────────────────────────────────
    def _print_result(self, e):
        tier_icons = {'CRITICAL', 'HIGH', 'MODERATE', 'LOW'}
        icon = tier_icons.get(e['risk_tier'])
        print(f"  {e['district']:<20} {icon}  {e['risk_tier']}")

print('PovertyRiskModel class defined.')

PovertyRiskModel class defined.


In [ ]:
# ── Instantiate the unified model ──────────────────────────────────────────
model = PovertyRiskModel(
    scaler=scaler,
    feature_cols=extended_features,
    rules=ALLOCATION_RULES,
    encoder=None,              # IMPORTANT: do not store encoder
    district_data=df,
    embeddings=corpus_embeddings
)

print('PovertyRiskModel instantiated successfully.')
print(f'   Districts loaded: {len(df)}')
print(f'   Features used:    {len(extended_features)}')
print(f'   Embedding dim:    {corpus_embeddings.shape[1]}')
print(f'   Risk tiers:       {df["risk_tier"].value_counts().to_dict()}')

PovertyRiskModel instantiated successfully.
   Districts loaded: 25
   Features used:    15
   Embedding dim:    384
   Risk tiers:       {'CRITICAL': 10, 'HIGH': 9, 'MODERATE': 5, 'LOW': 1}


## 💾 Step 6 — Save Model (Single Pickle)

In [ ]:
import pickle
import os
# MODEL_PATH = 'poverty_risk_model.pkl'

# with open(MODEL_PATH, 'wb') as f:
#     pickle.dump(model, f)

# import os
# size_mb = os.path.getsize(MODEL_PATH) / 1e6
# print(f' Model saved → {MODEL_PATH}  ({size_mb:.1f} MB)')

MODEL_PATH = "poverty_risk_model.pkl"

with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)

size_mb = os.path.getsize(MODEL_PATH) / 1e6
print(f" Model saved → {MODEL_PATH} ({size_mb:.1f} MB)")

# Download
files.download(MODEL_PATH)


# Download from Colab
files.download(MODEL_PATH)

 Model saved → poverty_risk_model.pkl (0.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🔄 Step 7 — Load & Verify Saved Model

In [ ]:
print("PovertyRiskModel instantiated successfully.")
print(f"Districts loaded: {len(df)}")
print(f"Features used: {len(extended_features)}")
print(f"Embedding dim: {corpus_embeddings.shape[1]}")
print(f"Risk tiers: {df['risk_tier'].value_counts().to_dict()}")

PovertyRiskModel instantiated successfully.
Districts loaded: 25
Features used: 15
Embedding dim: 384
Risk tiers: {'CRITICAL': 10, 'HIGH': 9, 'MODERATE': 5, 'LOW': 1}
